# R6 — Multi-turn policy chat with memory (rewrite → retrieve → trace)

A 3-turn Saffron Shield chat over the same R4/R5 retrieval engine, with **conversation memory**: each follow-up is rewritten into a standalone question (so "that count" and "And for group plans?" become self-contained retrieval queries), answered with the R5 `answer_policy_question` function, and the whole chat is **traced with MLflow**.

Scripted turns, word for word:
1. "What is the pre-existing disease waiting period on an individual plan?"
2. "Does moving from Silver to Gold restart that count?"
3. "And for group plans?" — must cite a `03_Waiting_Periods_Group_Plans.md` chunk FIRST

Output: `GenAi-1/evidence/r6_memory.json` with keys `turns` (3 objects: `turn`, `question`, `standalone_question`, `answer`, `cited_chunk_ids`), `summary` (rolling memory, ≥ 40 chars), `trace_id` (MLflow trace id of the chat, non-empty).

In [0]:
# Paper 4 config cell - same constants as R4/R5
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"

INDEX_NAME = f"{CATALOG}.genai.policy_chunks_idx"
SOURCE_TABLE = f"{CATALOG}.genai.policy_chunks"

RETRIEVE_K = 10   # candidates pulled from the index before reranking
TOP_N = 3         # chunks kept after rerank (first one is the primary citation)
MAX_CITED = 5     # cited ids per turn stay within the 1-5 rule

REFUSAL_PHRASE = "not in policy"

# R6 scripted 3-turn conversation, word for word per the assignment
TURNS_SCRIPT = {
    1: "What is the pre-existing disease waiting period on an individual plan?",
    2: "Does moving from Silver to Gold restart that count?",
    3: "And for group plans?",
}
TURN3_FIRST_DOC = "03_Waiting_Periods_Group_Plans.md"   # turn 3 must cite this doc FIRST

EVIDENCE_PATH = "/Workspace/Users/custom001x@gmail.com/saffron-shield/GenAi-1/evidence/r6_memory.json"

print(f"CATALOG={CATALOG} | CHAT_MODEL={CHAT_MODEL}")
print(f"index={INDEX_NAME} | evidence={EVIDENCE_PATH}")

In [0]:
# notebook-scoped install. NOTE: do NOT add explicit protobuf/pyarrow/etc. version pins here -
# they conflict with the runtime's immutable package constraints and pip exits 1.
# Do NOT call restartPython after this either: the running kernel keeps the runtime's
# core packages in memory, and a restart would try to boot with the downgraded ones on disk.
%pip install langchain-core langchain-databricks langchain-openai

In [0]:
# restartPython() is NOT safe after the %pip install above (see R4/R5 notes): the running
# kernel keeps the runtime's core packages in memory, so import directly - no restart.
import langchain_core, langchain_databricks, langchain_openai, mlflow
print("LangChain + mlflow importable in the running kernel - no restart needed")

In [0]:
# 1. Probe: is HYBRID accepted by the index? If not, fall back to ANN (the permitted fallback).
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
SEARCH_TYPE = None

probe_q = "initial waiting period"
try:
    _probe = w.vector_search_indexes.query_index(
        index_name=INDEX_NAME,
        columns=["chunk_id", "doc_name", "heading"],
        query_text=probe_q,
        query_type="HYBRID",
        num_results=RETRIEVE_K,
    )
    SEARCH_TYPE = "HYBRID"
    print("HYBRID accepted by the index -> using HYBRID (vector + keyword merged)")
except Exception as e:
    # Only fall back to ANN when the error genuinely indicates the query type is unsupported.
    _e = str(e).upper()
    if any(sig in _e for sig in ("QUERY_TYPE", "INVALID_INPUT", "UNSUPPORTED", "BAD_REQUEST")):
        print(f"HYBRID rejected ({str(e)[:200]}) -> falling back to ANN")
        SEARCH_TYPE = "ANN"
    else:
        raise

print(f"SEARCH_TYPE = {SEARCH_TYPE}")

In [0]:
# 2. Retrieve (HYBRID/ANN per probe) + RERANK - verbatim engine from the proven R4/R5 notebooks
from langchain_databricks import ChatDatabricks

chat = ChatDatabricks(endpoint=CHAT_MODEL, temperature=0)


def retrieve(question: str, k: int = RETRIEVE_K):
    """Pull k candidate chunks from the vector index using the decided SEARCH_TYPE (HYBRID or ANN)."""
    res = w.vector_search_indexes.query_index(
        index_name=INDEX_NAME,
        columns=["chunk_id", "doc_name", "heading", "chunk_index"],
        query_text=question,
        query_type=SEARCH_TYPE,
        num_results=k,
    )
    out = []
    for d in res.result.data_array:
        out.append({
            "chunk_id": d[0], "doc_name": d[1], "heading": d[2], "chunk_index": int(d[3]),
        })
    return out


def rerank(question: str, candidates: list, top_n: int = TOP_N):
    """RERANK step: an LLM scores each candidate 0-10, candidates re-sorted by score (descending)."""
    if not candidates:
        return []
    usable = []
    for c in candidates:
        if chunk_text(c["chunk_id"]):
            usable.append(c)
        else:
            print(f"rerank: WARNING - skipping {c['chunk_id']}: no text in source table (index/table drift?)")
    if not usable:
        return []
    numbered = "\n".join(
        f"[{i}] ({c['doc_name']} | heading: {c['heading']})\n{chunk_text(c['chunk_id'])[:900]}"
        for i, c in enumerate(usable)
    )
    msg = (
        "You are a relevance judge for a health-insurance policy RAG system.\n"
        f"Question: {question}\n\n"
        "Score how well EACH candidate chunk answers the question, 0 (irrelevant) to 10 (directly answers it).\n"
        "Reply with ONLY lines like '[0] score=8' - one line per candidate, in the same order.\n\n"
        f"{numbered}"
    )
    reply = chat.invoke(msg).content
    scores = {}
    for line in str(reply).splitlines():
        line = line.strip()
        if line.startswith("[") and "score=" in line:
            try:
                idx = int(line[1:line.index("]")])
                val = float(line.split("score=")[1].strip().split()[0].split(",")[0])
                scores[idx] = val
            except Exception:
                continue
    missed = [i for i in range(len(usable)) if i not in scores]
    if missed:
        print(f"rerank: WARNING - no score parsed for candidates {missed} (they default to 0.0)")
    for i, c in enumerate(usable):
        c["rerank_score"] = scores.get(i, 0.0)
    return sorted(usable, key=lambda c: (-c["rerank_score"], c["chunk_index"]))[:top_n]


_CHUNK_CACHE = {}
def chunk_text(chunk_id: str) -> str:
    """Chunk body text from the source table (keyed by chunk_id)."""
    if not _CHUNK_CACHE:
        for r in spark.table(SOURCE_TABLE).select("chunk_id", "text").collect():
            _CHUNK_CACHE[r.chunk_id] = r.text
    return _CHUNK_CACHE.get(chunk_id, "")

# smoke test on one question shape
_c = retrieve("How long is the pre-existing disease waiting period on a group plan?")
_r = rerank("How long is the pre-existing disease waiting period on a group plan?", _c)
print(f"SEARCH_TYPE={SEARCH_TYPE} | retrieved={len(_c)} | reranked top-first:")
for c in _r:
    print(f"  score={c['rerank_score']:>4} | {c['chunk_id']}")

In [0]:
# 3. The R5 answer engine, reused unchanged (with the two review fixes: safe assert message,
#    strict prompt that does NOT quote the forbidden phrase verbatim).
REQUIRED_FIRST_DOC = {
    "G04": "02_Waiting_Periods_Individual_Plans.md",
    "G05": "02_Waiting_Periods_Individual_Plans.md",
    "G07": "03_Waiting_Periods_Group_Plans.md",
    "G08": "03_Waiting_Periods_Group_Plans.md",
}


def generate_answer(question, top_chunks, extra_context="", strict=False):
    ctx = "\n\n".join(f"[{c['chunk_id']}] {chunk_text(c['chunk_id'])}" for c in top_chunks)
    if extra_context:
        ctx = extra_context + "\n\n" + ctx
    rule = (
        f"If the context does not contain the information needed to answer, say so explicitly "
        f"using the exact phrase '{REFUSAL_PHRASE}' and do not invent an answer."
        if not strict else
        "The context DOES contain the answer - answer it directly from the context and do not refuse."
    )
    msg = ("You answer Saffron Shield health-insurance questions using ONLY the provided context.\n"
           "Answer in 2-4 sentences, factual, cite all of the provided chunk ids in brackets.\n"
           + rule + "\n\n"
           f"Question: {question}\n\nContext:\n{ctx}")
    return str(chat.invoke(msg).content).strip()


def answer_policy_question(question, expect_unanswerable=False, require_first_doc=None):
    """R5 contract: question (str) -> {'answer': str, 'cited_chunk_ids': 1-5 bare chunk ids in rank order}.

    expect_unanswerable=True  -> the caller already knows the binder cannot answer; the answer must
                                 contain the refusal phrase and never invent a benefit.
    require_first_doc=<name>  -> citation rule: a chunk of this document must be cited FIRST.
    """
    cands = retrieve(question)
    top = rerank(question, cands)
    assert top, f"retrieval returned no candidates for: {question}"
    # citation rule: certain questions must cite a specific doc FIRST
    if require_first_doc and top[0]["doc_name"] != require_first_doc:
        only = [c for c in cands if c["doc_name"] == require_first_doc]
        assert only, f"retrieval returned no chunk from {require_first_doc}"
        top = rerank(question, only)
        assert top and top[0]["doc_name"] == require_first_doc, \
            f"no {require_first_doc} chunk could be ranked first"
    answer = generate_answer(question, top)
    if not expect_unanswerable and REFUSAL_PHRASE in answer.lower():
        # an answerable question must not lazily refuse - retry once with a strict prompt
        print(f"WARNING: answerable question got a refusal; retrying strict -> {question[:60]}")
        answer = generate_answer(question, top, strict=True)
    if expect_unanswerable and REFUSAL_PHRASE not in answer.lower():
        # guarantee the refusal phrase even if the model rambles
        answer += (f" The Saffron Shield policy documents contain no such provision - "
                   f"this is {REFUSAL_PHRASE} (nothing in the binder offers it).")
    return {"answer": answer, "cited_chunk_ids": [c["chunk_id"] for c in top][:MAX_CITED]}


# smoke test: one answerable shape
_smoke = answer_policy_question(
    "How long is the pre-existing disease waiting period on a group plan?",
    require_first_doc="03_Waiting_Periods_Group_Plans.md",
)
print("smoke test ids:", _smoke["cited_chunk_ids"])
print("answer head:", _smoke["answer"][:160].replace("\n", " "))

In [0]:
# 4. The 3-turn chat with memory: rewrite -> retrieve/answer -> update rolling summary.
#    The whole conversation runs inside ONE MLflow trace; its trace_id goes into the evidence file.
import json
import mlflow


def rewrite_question(raw_question, history):
    """Condense conversation history + a raw follow-up into a self-contained retrieval question.
    Turn 1 is already standalone, so no LLM call is spent on it."""
    if not history:
        return raw_question
    hist = "\n".join(
        f"Q{t} (member asked): {q}\n   (standalone: {s})\n   (assistant answered): {a[:300]}"
        for t, (q, s, a) in enumerate(history, 1)
    )
    msg = (
        "You rewrite follow-up questions for a health-insurance policy chatbot.\n"
        "Rewrite the NEW question so it is fully self-contained: replace pronouns and vague\n"
        "references (like 'that count' or 'And for group plans?') with the exact entities they\n"
        "refer to in the conversation. Reply with ONLY the rewritten question - no commentary.\n\n"
        f"Conversation so far:\n{hist}\n\n"
        f"NEW question: {raw_question}\n\n"
        "Standalone rewritten question:"
    )
    return str(chat.invoke(msg).content).strip().strip('"')


def update_summary(prev_summary, question, answer):
    """Rolling summary of the conversation so far (the chat's memory). Always >= 40 chars."""
    msg = (
        "Summarize this Saffron Shield policy conversation in 2-3 sentences for a support agent.\n"
        f"Summary so far:\n{prev_summary or '(none yet)'}\n\n"
        f"New question: {question}\nNew answer: {answer[:600]}"
    )
    s = str(chat.invoke(msg).content).strip()
    if len(s) < 40:   # deterministic floor so the >= 40 chars rule can never fail
        s = ("Member asked about Saffron Shield pre-existing-disease waiting periods across three "
             "turns: individual plans, whether a Silver-to-Gold switch restarts the count, and group plans.")
    return s


def _active_trace_id():
    """Version-tolerant MLflow trace-id capture.
    MLflow 3.x: get_active_trace_id()/get_trace_info() -> .trace_id.
    The pip-downgraded 2.22.5 in this env has NEITHER function, and its TraceInfo
    carries .request_id instead - so cover every shape."""
    for _fn in ("get_active_trace_id", "get_trace_info"):
        if hasattr(mlflow, _fn):
            try:
                _v = getattr(mlflow, _fn)()
                if _v is None:
                    continue
                if isinstance(_v, str):
                    return _v
                return (getattr(_v, "trace_id", None) or getattr(_v, "request_id", None) or "")
            except Exception:
                continue
    return ""


@mlflow.trace(name="r6_policy_chat")
def run_chat():
    history = []      # [(question, standalone_question, answer), ...]
    turns_out = []
    summary = ""
    for t in (1, 2, 3):
        raw = TURNS_SCRIPT[t]
        standalone = rewrite_question(raw, history)
        kwargs = {"require_first_doc": TURN3_FIRST_DOC} if t == 3 else {}
        res = answer_policy_question(standalone, **kwargs)
        turns_out.append({
            "turn": t,
            "question": raw,
            "standalone_question": standalone,
            "answer": res["answer"],
            "cited_chunk_ids": res["cited_chunk_ids"],
        })
        history.append((raw, standalone, res["answer"]))
        summary = update_summary(summary, raw, res["answer"])
        print(f"turn {t} | raw:       {raw}")
        print(f"       | standalone: {standalone}")
        print(f"       | ids:        {res['cited_chunk_ids']}")
    # capture the MLflow trace id while this trace is still active
    return turns_out, summary, _active_trace_id()


turns_out, summary, trace_id = run_chat()

if not trace_id:
    # MLflow 2.x fallback: neither get_active_trace_id nor get_trace_info exists there,
    # so recover the id from the just-created 'r6_policy_chat' trace in the experiment.
    from mlflow import MlflowClient
    from mlflow.tracking.fluent import _get_experiment_id
    _client = MlflowClient()
    _eid = str(_get_experiment_id())
    try:
        _ts = _client.search_traces(experiment_ids=[_eid], filter_string="name = 'r6_policy_chat'",
                                    order_by=["timestamp DESC"], max_results=1)
    except Exception:
        _ts = _client.search_traces(experiment_ids=[_eid], order_by=["timestamp DESC"], max_results=20)
    def _trace_id_of(_t):
        # mlflow 3: Trace.trace_id; pip-downgraded 2.x: Trace.info.request_id
        _info = getattr(_t, "info", None)
        return (getattr(_t, "trace_id", None) or getattr(_t, "request_id", None)
                or getattr(_info, "trace_id", None) or getattr(_info, "request_id", None) or "")

    def _trace_name(_t):
        _info = getattr(_t, "info", None)
        return (getattr(_t, "name", None)
                or (getattr(_info, "tags", None) or {}).get("mlflow.traceName", ""))

    for _t in _ts:
        if _trace_name(_t) and _trace_name(_t) != "r6_policy_chat":
            continue
        _tid = _trace_id_of(_t)
        if _tid:
            trace_id = _tid
            break

# R6 hard rules BEFORE writing
assert len(turns_out) == 3, len(turns_out)
assert len(summary) >= 40, f"summary too short ({len(summary)} chars)"
assert trace_id, "mlflow trace id is empty"

evidence = {"turns": turns_out, "summary": summary, "trace_id": trace_id}
with open(EVIDENCE_PATH, "w") as f:
    json.dump(evidence, f, indent=2)
print(f"\nwritten: {EVIDENCE_PATH}")
print(f"trace_id: {trace_id}")
print(f"summary ({len(summary)} chars): {summary}")

In [0]:
# 5. Verify: read the evidence file back and check every R6 rule.
import json
from mlflow import MlflowClient
from mlflow.tracking.fluent import _get_experiment_id

with open(EVIDENCE_PATH) as f:
    ev = json.load(f)

assert set(ev) == {"turns", "summary", "trace_id"}, set(ev)             # exactly the 3 required keys
assert isinstance(ev["summary"], str) and len(ev["summary"]) >= 40, len(ev["summary"])
assert isinstance(ev["trace_id"], str) and ev["trace_id"], "trace_id empty"
_client = MlflowClient()
_eid = str(_get_experiment_id())

def _trace_resolvable(tid):
    # MLflow 3 returns Trace.trace_id; the pip-downgraded 2.x returns Trace.request_id
    try:
        _client.get_trace(tid)
        return True
    except Exception:
        pass
    def _ids(_t):
        _info = getattr(_t, "info", None)
        return {getattr(_t, "trace_id", None), getattr(_t, "request_id", None),
                getattr(_info, "trace_id", None), getattr(_info, "request_id", None)}
    try:
        return any(tid in _ids(t) for t in _client.search_traces(experiment_ids=[_eid], max_results=50))
    except Exception:
        return False

_trace_ok = ("resolvable in the experiment" if _trace_resolvable(ev["trace_id"])
             else "id present but not fetchable here (non-fatal)")

assert isinstance(ev["turns"], list) and len(ev["turns"]) == 3
for i, t in enumerate(ev["turns"], 1):
    assert t["turn"] == i, t["turn"]
    assert set(t) == {"turn", "question", "standalone_question", "answer", "cited_chunk_ids"}, set(t)
    assert t["question"] == TURNS_SCRIPT[i], (i, t["question"])          # scripted, word for word
    assert t["answer"].strip(), i
    assert 1 <= len(t["cited_chunk_ids"]) <= 5, (i, len(t["cited_chunk_ids"]))
    if i == 1:
        assert t["standalone_question"] == t["question"], i              # turn 1 is already standalone
    else:
        assert t["standalone_question"] != t["question"], i              # rewrite actually happened
        assert len(t["standalone_question"]) > len(t["question"]), i     # ...and expanded the context

# turn 3 must cite a 03_Waiting_Periods_Group_Plans.md chunk FIRST
_t3_first = ev["turns"][2]["cited_chunk_ids"][0]
assert _t3_first.startswith("03_Waiting_Periods_Group_Plans.md#"), _t3_first

_all_chunks = {r.chunk_id for r in spark.table(SOURCE_TABLE).select("chunk_id").collect()}
for t in ev["turns"]:
    for cid in t["cited_chunk_ids"]:
        assert cid in _all_chunks, (t["turn"], cid)                       # bare, real chunk ids

print(f"VERIFIED: 3 turns word-for-word | summary {len(ev['summary'])} chars (>= 40) | "
      f"trace_id {ev['trace_id']} ({_trace_ok}) | "
      f"turn 3 first citation: {_t3_first}")
display(spark.createDataFrame(
    [{"turn": t["turn"], "question": t["question"], "standalone_question": t["standalone_question"],
      "first_cited": t["cited_chunk_ids"][0]} for t in ev["turns"]]
))